# Reading Renewals Data

In [0]:
df_renewal = spark.read.table("post_renewal_churn.raw.renewal_calls")

df_renewal = df_renewal.select(
    "Call_ID",
    "Call_Direction",
    "Co_Ref",
    "Call_Date",
    "Serious_Complaint",
    "Other_Complaint",
    "Discussion_on_Price_Increase",
    "Renewal_Impact_Due_to_Price_Increase",
    "Discount_or_Waiver_Requested",
    "Call_Reschedule_Request",
    "Explicit_Competitor_Mention",
    "Analysed_Call"
)

# Display Data

In [0]:
df_renewal.display()

# Merging billings and renewal calls data

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ── 1. Load / prepare billings ──────────────────────────────────────────
df_billings = spark.table("post_renewal_churn.raw.billings") \
    .filter(F.col("prospect_outcome") != "Open") \
    .withColumn("datediff", F.datediff("closed_date", "prospect_renewal_date")) \
    .filter(F.col("datediff") < 29) \
    .withColumn("index", F.monotonically_increasing_id())

# Use df_renewal instead of spark.table("post_renewal_churn.raw.renewal_calls")
df_calls = df_renewal \
    .filter(F.col("Analysed_Call") == "1") \
    .filter(F.col("Customer_Renewal_Response_Category") != "null") \
    .filter(F.col("Customer_Renewal_Response_Category") != "Not Mentioned")

# ── 3. Join on co_ref — explicit condition to avoid ambiguous reference ──
df_joined = df_billings.join(
    df_calls,
    on=df_billings["Co_Ref"] == df_calls["Co_Ref"],
    how="left"
).filter(
    (F.col("Call_Date") >= F.col("prospect_renewal_date")) &
    (F.col("Call_Date") <= F.col("closed_date"))
).drop(df_calls["Co_Ref"])

# ── 4. Keep only the LATEST call per billing row (index) ────────────────
window = Window.partitionBy("index").orderBy(F.desc("Call_Date"))

df_latest_call = df_joined \
    .withColumn("rn", F.row_number().over(window)) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

df_latest_call.display()
df_final = df_latest_call
df_final.write.mode("overwrite").saveAsTable("post_renewal_churn.raw.joined_two_tables")

# Count final data

In [0]:
df_latest_call.count()

# Hypothesis Testing

In [0]:


from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from scipy.stats import chi2_contingency, ttest_ind
import pandas as pd

# Columns to test
columns = [
    "Call_ID",
    "Call_Direction",
    "Co_Ref",
    "Call_Date",
    "Serious_Complaint",
    "Other_Complaint",
    "Discussion_on_Price_Increase",
    "Renewal_Impact_Due_to_Price_Increase",
    "Discount_or_Waiver_Requested",
    "Call_Reschedule_Request",
    "Explicit_Competitor_Mention",
    "Analysed_Call"
]

# Categorical columns for chi-square test
categorical_cols = [
    "Call_Direction",
    "Serious_Complaint",
    "Other_Complaint",
    "Discussion_on_Price_Increase",
    "Renewal_Impact_Due_to_Price_Increase",
    "Discount_or_Waiver_Requested",
    "Call_Reschedule_Request",
    "Explicit_Competitor_Mention",
    "Analysed_Call"
]

# Numeric columns for t-test (if any)
numeric_cols = []

# Prepare DataFrame for testing
df_test = df_latest_call.select(columns + ["Prospect_Outcome"])

results = []

for col in columns:
    h0 = f"There is no association between {col} and prospect_outcome."
    h1 = f"There is an association between {col} and prospect_outcome."
    if col in categorical_cols:
        # Chi-square test
        pdf = df_test.groupBy(col, "Prospect_Outcome").count().toPandas().pivot(index=col, columns="Prospect_Outcome", values="count").fillna(0)
        chi2, p, _, _ = chi2_contingency(pdf.values)
        result = "Reject H0" if p < 0.05 else "Fail to reject H0"
        results.append((col, h0, h1, p, result))
    elif col in numeric_cols:
        # T-test
        pdf = df_test.select(col, "Prospect_Outcome").toPandas()
        groups = pdf.groupby("Prospect_Outcome")[col]
        if len(groups) == 2:
            vals1 = groups.get_group(list(groups.groups.keys())[0]).dropna()
            vals2 = groups.get_group(list(groups.groups.keys())[1]).dropna()
            t_stat, p = ttest_ind(vals1, vals2)
            result = "Reject H0" if p < 0.05 else "Fail to reject H0"
            results.append((col, h0, h1, p, result))
        else:
            results.append((col, h0, h1, None, "Not enough groups for t-test"))
    else:
        results.append((col, h0, h1, None, "Not tested"))

# Print results
for col, h0, h1, p, result in results:
    print(f"Column: {col}")
    print(f"H0: {h0}")
    print(f"H1: {h1}")
    print(f"p-value: {p}")
    print(f"Result: {result}\n")

# Feature Selection

In [0]:
reject_h0_cols = [
    "Co_Ref",
    "Call_ID",
    "Call_Direction",
    "Call_Date",
    "Serious_Complaint",
    "Other_Complaint",
    "Renewal_Impact_Due_to_Price_Increase",
    "Discount_or_Waiver_Requested",
    "Explicit_Competitor_Mention",
    "Analysed_Call"
]

df_reject_h0 = df_latest_call.select(reject_h0_cols + ["Prospect_Outcome"])
display(df_reject_h0)

# Variables with Significant Impact on Prospect Outcome

In [0]:
reject_h0_cols = [
    "Co_Ref",
    "Call_ID",
    "Call_Direction",
    "Call_Date",
    "Serious_Complaint",
    "Other_Complaint",
    "Renewal_Impact_Due_to_Price_Increase",
    "Discount_or_Waiver_Requested",
    "Explicit_Competitor_Mention",
    "Analysed_Call"
]



# Selecting features needed for target

In [0]:
df = spark.read.table("post_renewal_churn.raw.renewal_calls")

df = df.select(
    "Co_Ref",
    "Call_ID",
    "Call_Direction",
    "Call_Date",
    "Serious_Complaint",
    "Other_Complaint",
    "Renewal_Impact_Due_to_Price_Increase",
    "Discount_or_Waiver_Requested",
    "Explicit_Competitor_Mention",
    "Analysed_Call"
)

# Serious Complaint Feature column values

In [0]:
display(df.select("Serious_Complaint").distinct())
df = df.withColumn('Serious_Complaint', F.when(F.col("Serious_Complaint").isNull(), "UnKnown").otherwise(F.col("Serious_Complaint")))

# Filling null values for other complaint feature

In [0]:
df = df.withColumn('Other_Complaint', F.when(F.col("Other_Complaint").isNull(), "UnKnown").otherwise(F.col("Other_Complaint")))

# Unique values of other complaint feature

In [0]:
display(df.select("Other_Complaint").distinct())

# Renewal Impact unique values

In [0]:
display(df.select("Renewal_Impact_Due_to_Price_Increase").distinct())

# Filling null values of renewal impact feature

In [0]:
df = df.withColumn(
    'Renewal_Impact_Due_to_Price_Increase',
    F.when(F.col("Renewal_Impact_Due_to_Price_Increase").isNull(), "UnKnown")
     .when(F.col("Renewal_Impact_Due_to_Price_Increase").rlike(r"(?i)^\s*no\s*$|^\s*\*\*no\*\*\s*$"), "No")
     .when(F.col("Renewal_Impact_Due_to_Price_Increase").rlike(r"(?i)^\s*yes\s*$|^yes\s*\(.*\)$|^yes,.*$|^yes\s*due.*$"), "Yes")
     .when(F.col("Renewal_Impact_Due_to_Price_Increase").rlike(r"(?i)^\s*n/a\s*$|^\s*not applicable\s*$"), "UnKnown")
     .when(F.col("Renewal_Impact_Due_to_Price_Increase").rlike(r"(?i)^\s*\[yes/no\]\s*$"), "Yes/No")
     .otherwise(F.col("Renewal_Impact_Due_to_Price_Increase"))
)

# Unique values of renewal impact feature

In [0]:
display(df.select("Renewal_Impact_Due_to_Price_Increase").distinct())

# Unique values of discount waiver feature

In [0]:
display(df.select("Discount_or_Waiver_Requested").distinct())

# Filling null values of Discount Waiver feature

In [0]:
df = df.withColumn(
    'Discount_or_Waiver_Requested',
    F.when(F.col("Discount_or_Waiver_Requested").isNull(), "UnKnown")
     .when(F.col("Discount_or_Waiver_Requested").rlike(r"(?i)^\s*no\s*$|^\s*\*\*no\*\*\s*$|no\s*\(.*\)|no,.*|no\s*but.*|no\s*although.*|no\s*the.*|no\s*replaced.*"), "No")
     .when(F.col("Discount_or_Waiver_Requested").rlike(r"(?i)^\s*yes\s*$|^yes\s*\(.*\)$|^yes,.*$|yes\s*the.*|yes\s*customer.*|yes\s*agent.*"), "Yes")
     .when(F.col("Discount_or_Waiver_Requested").rlike(r"(?i)^\s*n/a\s*$"), "N/A")
     .when(F.col("Discount_or_Waiver_Requested").rlike(r"(?i)^\s*not applicable\s*$"), "Not applicable")
     .when(F.col("Discount_or_Waiver_Requested").rlike(r"(?i)^\s*\[yes/no\]\s*$"), "Yes/No")
     .when(F.col("Discount_or_Waiver_Requested").rlike(r"The customer was not asking for a discount on a price increase, but was instead confirming and proceeding with a previously negotiated price from last year's discussion."), "No")
     .otherwise(F.col("Discount_or_Waiver_Requested"))
)

# Cleaning values of Discount Waiver feature

In [0]:
df = df.withColumn(
    'Discount_or_Waiver_Requested',
    F.when(F.col("Discount_or_Waiver_Requested")=='N/A', "UnKnown").otherwise(F.col("Discount_or_Waiver_Requested"))
)

# Unique values in discount waiver feature

In [0]:
display(df.select("Discount_or_Waiver_Requested").distinct())

# Unique values in explicit competitor mention feature

In [0]:

display(df.select("Explicit_Competitor_Mention").distinct())

# Filling cleaned values for Explicit_Competitor_Mention feature

In [0]:
df = df.withColumn(
    'Explicit_Competitor_Mention',
    F.when(F.col("Explicit_Competitor_Mention").isNull(), "UnKnown")
     .when(F.col("Explicit_Competitor_Mention")=="XXXX", "UnKnown")
     .when(F.col("Explicit_Competitor_Mention")=="Prompt 2:", "UnKnown")
     .when(F.col("Explicit_Competitor_Mention")=="UNAVAILABLE", "UnKnown")
     .when(F.col("Explicit_Competitor_Mention").rlike(r"(?i)^\s*yes\s*$"), "Yes")
     .otherwise("No")
)

display(df.select("Explicit_Competitor_Mention").distinct())

# Writing Cleaned data into table

In [0]:
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("post_renewal_churn.raw.renewal_call_cleaned")

# Displaying final cleaned data

In [0]:
%sql
select * from post_renewal_churn.raw.renewal_call_cleaned;